# 第9章 可赎回债券与可回售债券 — 编程实验完整解答

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/solutions/ch09_solutions.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/solutions/ch09_solutions.ipynb)

本 notebook 给出本章全部编程实验的完整可运行解答；联网（akshare）部分以注释/降级方式给出，离线也能跑通。


In [ ]:
# 自举单元：Colab/Binder 自动安装 fi；本地跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','--depth','1','https://github.com/albertandking/fixed-income.git','/content/fi-book'],check=False)
        subprocess.run([sys.executable,'-m','pip','install','-e','/content/fi-book'],check=False)
    else:
        print('提示：仓库根目录执行 `uv sync --extra all` 后运行本 notebook。')


## 编程实验 7：例10.1/10.2 + 图9-1（负凸性）


In [ ]:
import numpy as np
from fi import tree, plotting
plotting.use_chinese_style()
t = tree.short_rate_tree(0.03, 0.20, 6)
s = tree.value_bond(t, 6.0, 100); c = tree.value_bond(t, 6.0, 100, call_price=100, call_from=1)
print(f'普通债={s:.4f} 可赎回={c:.4f} 赎回期权={s-c:.4f}')
print(f'有效久期: 普通={tree.effective_duration_tree(0.03,0.20,6,6.0):.4f} 可赎回={tree.effective_duration_tree(0.03,0.20,6,6.0,call_price=100,call_from=1):.4f}')
refs = np.linspace(0.01,0.08,36)
fig, ax = plotting.new_axes()
ax.plot(refs*100, [tree.value_bond(tree.short_rate_tree(r,0.20,6),6.0,100) for r in refs], label='普通债')
ax.plot(refs*100, [tree.value_bond(tree.short_rate_tree(r,0.20,6),6.0,100,call_price=100,call_from=1) for r in refs], label='可赎回债')
ax.axhline(100, ls=':', color='gray'); ax.set_xlabel('短期利率 r0 (%)'); ax.set_ylabel('价格'); ax.set_title('可赎回债低利率端被封顶(负凸性)'); ax.legend(); fig.tight_layout()


## 编程实验 8：有效凸性扫描，找负凸性区间


In [ ]:
for r in [0.02,0.03,0.04,0.05,0.06,0.07]:
    ec = tree.effective_convexity_tree(r,0.20,6,6.0,dy=2e-3,call_price=100,call_from=1)
    print(f'r0={r:.0%}: 有效凸性={ec:9.2f}' + ('  <-- 负凸性(期权价内附近)' if ec<0 else ''))


## 编程实验 9：QuantLib OAS 对波动率的敏感性


In [ ]:
import QuantLib as ql
today = ql.Date(15,6,2026); ql.Settings.instance().evaluationDate = today
dc = ql.ActualActual(ql.ActualActual.ISDA); ts = ql.YieldTermStructureHandle(ql.FlatForward(today,0.03,dc))
sched = ql.Schedule(today, today+ql.Period(6,ql.Years), ql.Period(ql.Annual), ql.NullCalendar(), ql.Unadjusted, ql.Unadjusted, ql.DateGeneration.Backward, False)
calls = ql.CallabilitySchedule()
for y in range(1,6): calls.append(ql.Callability(ql.BondPrice(100.0, ql.BondPrice.Clean), ql.Callability.Call, today+ql.Period(y,ql.Years)))
bond = ql.CallableFixedRateBond(0,100.0,sched,[0.06],dc,ql.Unadjusted,100.0,today,calls)
bond.setPricingEngine(ql.TreeCallableFixedRateBondEngine(ql.HullWhite(ts,0.03,0.015),100)); mkt = bond.cleanPrice()
print(f'市价={mkt:.4f}')
for vol in (0.005,0.015,0.03):
    bond.setPricingEngine(ql.TreeCallableFixedRateBondEngine(ql.HullWhite(ts,0.03,vol),100))
    oas = bond.OAS(mkt, ts, dc, ql.Compounded, ql.Annual, today, 1e-10, 200, 0.0)*1e4
    print(f'vol={vol*100:.1f}%: OAS={oas:.2f}bp')
print('OAS 随波动率上升而下降')
